# 1. roboflow 서버의 모델을 통한 예측

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일 읽기

rf_api_key = os.getenv("RF_API_KEY")

# print(rf_api_key)

In [3]:
from inference_sdk import InferenceHTTPClient
CLIENT = InferenceHTTPClient(
api_url="https://detect.roboflow.com",
api_key= rf_api_key
)
filename = './images/number1.jpg'
result = CLIENT.infer(filename, model_id="numbers-doqnw/3")
print(result, "\n")

from pprint import pprint
# pprint(result['predictions'])
pprint([result['predictions'][i]['class'] for i in range(len(result['predictions']))])

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


{'inference_id': 'd3cfa3a4-2eea-4b3b-93ed-728fe7757728', 'time': 0.0061004930175840855, 'image': {'width': 416, 'height': 416}, 'predictions': [{'x': 194.5, 'y': 278.5, 'width': 27.0, 'height': 45.0, 'confidence': 0.9962424039840698, 'class': '6', 'class_id': 6, 'detection_id': '88a44457-8b09-4873-857c-a885c8fd139e'}, {'x': 118.5, 'y': 193.5, 'width': 33.0, 'height': 43.0, 'confidence': 0.9831866025924683, 'class': '4', 'class_id': 4, 'detection_id': '1492f4a7-b977-4b0d-a034-463d1ec09d8f'}, {'x': 167.5, 'y': 362.0, 'width': 11.0, 'height': 14.0, 'confidence': 0.8760370016098022, 'class': '6', 'class_id': 6, 'detection_id': 'f59c2180-5055-4070-8aa2-700ddd9e4672'}, {'x': 378.0, 'y': 330.5, 'width': 30.0, 'height': 35.0, 'confidence': 0.8259488344192505, 'class': '3', 'class_id': 3, 'detection_id': 'f5f4fbd0-40e3-4751-9499-2e068513afcc'}, {'x': 266.0, 'y': 314.0, 'width': 28.0, 'height': 46.0, 'confidence': 0.8144934177398682, 'class': '3', 'class_id': 3, 'detection_id': '70f543b6-9535-48

## 바운딩 박스 그리기

In [4]:
import cv2
filename = './images/number1.jpg'
img = cv2.imread(filename)
for pred in result["predictions"]:
    x, y = pred['x'], pred['y']
    width, height = pred['width'], pred['height']
    conf = pred['confidence']
    cls = pred['class']
    x1, y1 = int(x-width/2), int(y-height/2)
    x2, y2 = int(x+width/2), int(y+height/2)
    cv2.rectangle(img, (x1,y1), (x2, y2), (0,0,255), 2)
    cv2.putText(img, f'{cls} {conf:.4f}', (x1,y1), cv2.FONT_HERSHEY_PLAIN, 1, (0,0,255))

cv2.imshow('image', img)
cv2.waitKey(0)    
cv2.destroyAllWindows()

## roboflow 패키지의 Model 객체를 통한 예측

### ./images/number2.jpg 추론하기

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)

# 기본 워크스페이스 조회 → 그 안에서 "numbers-doqnw" 프로젝트 선택
project = rf.workspace().project("numbers-doqnw")

# 버전 3의 학습된 모델 객체 (confidence=40, overlap=30, format='json' 기본값)
model = project.version(3).model

# 추론할 로컬 이미지 경로
filename = './images/number2.jpg'

# 이미지를 base64로 인코딩해 Roboflow 서버로 POST → 결과를 dict로 변환
result = model.predict(filename).json()

retrying...
retrying...
loading Roboflow workspace...
loading Roboflow project...


version.model is deprecated and will be removed in a future release; use version.models() (all trained models) or version.trainings() instead.


### 바운딩 박스 그리기

In [9]:
import cv2

# 이미지 파일 읽기
img = cv2.imread(filename)

# 이미지의 세로(height), 가로(width) 추출
img_height, img_width = img.shape[:2]

# YOLO 모델이 640x640 기준으로 예측했을 경우 스케일 비율 계산
x_ratio, y_ratio = 640 / img_width, 640 / img_height
print(img_width, img_height, x_ratio, y_ratio)

# 예측된 객체(predictions) 반복 처리
for pred in result["predictions"]:

    # 중심 좌표(x, y), 박스 크기(width, height)
    x, y = pred['x'], pred['y']
    w, h = pred['width'], pred['height']

    # 클래스 이름(class), 신뢰도(confidence)
    cls, conf = pred['class'], pred['confidence']

    # YOLO 형식(center x, center y, width, height)을
    # 좌측상단(x1,y1), 우측하단(x2,y2) 좌표로 변환
    x1 = int(x - w/2)
    y1 = int(y - h/2)
    x2 = int(x + w/2)
    y2 = int(y + h/2)

    # 바운딩박스 그리기
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # 클래스명 + 신뢰도 표시
    cv2.putText(
        img, 
        f'{cls} {conf:.4f}',    # ex) digit 0.9876
        (x1, y1),               # 텍스트 위치 = 박스 좌상단
        cv2.FONT_HERSHEY_PLAIN, # 폰트
        1,                      # 폰트 크기
        (0, 0, 255)             # 빨간색
    )

# 이미지 출력 (창 닫기 전까지 대기)
cv2.imshow('image', img)
cv2.waitKey(0)

# 모든 OpenCV 창 닫기
cv2.destroyAllWindows()


800 453 0.8 1.4128035320088301


### ./images/number1.jpg 추론하기

In [10]:
from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)
project = rf.workspace().project("numbers-doqnw")
model = project.version(3).model

filename = './images/number1.jpg'
result = model.predict(filename).json()

loading Roboflow workspace...
loading Roboflow project...


version.model is deprecated and will be removed in a future release; use version.models() (all trained models) or version.trainings() instead.


### 바운딩박스 그리기

In [12]:
import cv2

img = cv2.imread(filename)
img_height, img_width = img.shape[:2]
x_ratio, y_ratio = 640/img_width, 640/img_height
print(img_width, img_height, x_ratio, y_ratio)
for pred in result["predictions"]:
    # 중심 좌표(x, y), 박스 크기(width, height)
    x, y = pred['x'], pred['y']
    width, height = pred['width'], pred['height']

    # 클래스 이름(class), 신뢰도(confidence)
    conf, cls = pred['confidence'], pred['class']

    # YOLO 형식(center x, center y, width, height)을
    # 좌측상단(x1,y1), 우측하단(x2,y2) 좌표로 변환
    x1, y1 = int(x-width/2), int(y-height/2)
    x2, y2 = int(x+width/2), int(y+height/2)
    x1, x2 = int(x1*x_ratio), int(x2*x_ratio)
    y1, y2 = int(y1*y_ratio), int(y2*y_ratio)
    
     # 바운딩박스 그리기
    cv2.rectangle(img, (x1,y1), (x2, y2), (0,0,255), 2)

    # 클래스명 + 신뢰도 표시
    cv2.putText(img, 
                f'{cls} {conf:.4f}',     # ex) digit 0.9876
                (x1,y1),                 # 텍스트 위치 = 박스 좌상단
                cv2.FONT_HERSHEY_PLAIN,  # 폰트
                1,                       # 폰트 크기
                (0,0,255))               # 빨간색 (BGR)
cv2.imshow('image', img)
cv2.waitKey(0)
cv2.destroyAllWindows()


416 416 1.5384615384615385 1.5384615384615385
